In [1]:
!python -V

Python 3.12.1


In [2]:
import mlflow
import pickle

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [3]:
# mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-exp")

2025/06/01 04:10:16 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-exp' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03-orchestration/artifacts/1', creation_time=1748751016809, experiment_id='1', last_update_time=1748751016809, lifecycle_stage='active', name='nyc-taxi-exp', tags={}>

In [5]:
def read_clean_df(filename):
    df = pd.read_parquet(filename)
    
    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration'] = df['duration'].apply(lambda x: x.total_seconds() / 60)
    
    df = df.loc[((df.duration >= 1) & (df.duration <= 60))]
    
    cat_feats = ['PULocationID', 'DOLocationID']
    df[cat_feats] = df[cat_feats].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    return df

In [9]:
df1 = read_clean_df('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-01.parquet')
df2 = read_clean_df('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-02.parquet')
df3 = read_clean_df('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-03.parquet')

In [10]:
len(df1.columns)

22

In [11]:
cat_feats = ['PULocationID', 'DOLocationID']
train_dicts = df1[cat_feats].to_dict(orient='records')

In [12]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [13]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 131892 stored elements and shape (65946, 467)>

In [14]:
target = 'duration'
y_train = df1[target].values
y_train

array([11.01666667,  6.76666667,  6.33333333, ..., 16.        ,
       18.        , 16.        ], shape=(65946,))

In [15]:
valid_dicts = df2[cat_feats].to_dict(orient='records')

In [16]:
X_valid = dv.transform(valid_dicts)

In [17]:
y_valid = df2[target].values
y_valid

array([19.58333333, 17.55      , 23.71666667, ..., 17.        ,
       17.        ,  5.        ], shape=(62574,))

In [43]:
mlflow.xgboost.autolog(disable=True)

In [22]:
from pathlib import Path
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [23]:
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_valid, label=y_valid)
    params = {
        'learning_rate':0.18772960331122643,
        'max_depth':84,
        'min_child_weight':4.414579549205784,
        'objective':'reg:linear',
        'reg_alpha': 0.05803027983214606,
        'reg_lambda':0.028651500789683607,
        'seed':42
    }
    
    mlflow.log_params(params)
    
    booster = xgb.train(
        params=params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, "validation")],
        early_stopping_rounds = 50
    )

    
        
    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_valid, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", 'wb') as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")


[0]	validation-rmse:8.43648
[1]	validation-rmse:7.80802


/usr/local/python/3.12.1/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [04:18:55] WARNING: /workspace/src/objective/regression_obj.cu:250: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[2]	validation-rmse:7.34776
[3]	validation-rmse:7.00650
[4]	validation-rmse:6.78095
[5]	validation-rmse:6.60366
[6]	validation-rmse:6.47681
[7]	validation-rmse:6.39742
[8]	validation-rmse:6.32803
[9]	validation-rmse:6.26343
[10]	validation-rmse:6.23333
[11]	validation-rmse:6.18642
[12]	validation-rmse:6.16359
[13]	validation-rmse:6.14520
[14]	validation-rmse:6.13668
[15]	validation-rmse:6.12160
[16]	validation-rmse:6.10653
[17]	validation-rmse:6.09705
[18]	validation-rmse:6.08883
[19]	validation-rmse:6.08241
[20]	validation-rmse:6.07642
[21]	validation-rmse:6.07295
[22]	validation-rmse:6.06902
[23]	validation-rmse:6.06572
[24]	validation-rmse:6.05921
[25]	validation-rmse:6.05356
[26]	validation-rmse:6.04954
[27]	validation-rmse:6.04664
[28]	validation-rmse:6.03644
[29]	validation-rmse:6.03200
[30]	validation-rmse:6.02992
[31]	validation-rmse:6.02766
[32]	validation-rmse:6.02276
[33]	validation-rmse:6.02232
[34]	validation-rmse:6.02136
[35]	validation-rmse:6.01686
[36]	validation-rmse:6

/usr/local/python/3.12.1/lib/python3.12/site-packages/mlflow/xgboost/__init__.py:168: UserWarning: [04:19:58] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)
2025/06/01 04:20:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run treasured-worm-855 at: http://localhost:5000/#/experiments/1/runs/ae39aa2fc1a7416eaf5846472eb8a715
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [24]:
with open('models/xgb_reg.bin', 'wb') as f_out:
    pickle.dump((dv, booster), f_out)